tactical players    
attack/aggressive/tactical

In [9]:
import pandas as pd
import numpy as np
from sklearn.svm import OneClassSVM
from sklearn.preprocessing import StandardScaler

In [10]:
tactical_final_list = [
    # --- Top Tier (2700+) ---
    'Hikaru Nakamura', 'Alireza Firouzja', 'Daniil Dubov', 'Shakhriyar Mamedyarov',
    'Maxime Vachier-Lagrave', 'Ian Nepomniachtchi', 'Jan-Krzysztof Duda',
    'Alexander Grischuk', 'Vladimir Fedoseev', 'Parham Maghsoodloo', 'Arjun Erigaisi',

    # --- High Tactical/Sharp (2600-2700) ---
    'Salem AR Saleh', 'Javokhir Sindarov', 'Gawain Maroroa Jones', 'Alex Fier',
    'Mustafa Yilmaz','Vasif Durarbayli', 'Denis Khismatullin',
    'Vitaliy Bernadskiy','Volodymyr Onyshchuk','Thomas Beerdsen','Bassem Amin'
    # Vitaliy Bernadskiy ; Volodymyr Onyshchuk ; Thomas Beerdsen,'Bassem Amin'

    # --- Mid/Low Tier & Online Specialists (Aggressive Style) ---
    'Baadur Jobava', 'Simon Williams', 'Levy Rozman', 'Hans Niemann',
    'Daniel Naroditsky', 'Andrew Tang', 
    'Aman Hambleton', 'Eric Hansen', 'Yaacov Norowitz', 'Tanitoluwa Adewumi',
    'Bibisara Assaubayeva', 'Novendra Priasmoro',
    'Jose Fernando Cuenca',
    'Jose Martinez','Tuan Minh Le','Vaishali R','Robert L. Hess','Eric Rosen'
    #Eric Rosen ; Jose Martinez ；Tuan Minh Le； Vaishali R； Robert L. Hess； Emilio Cordova
    # 'Eline Roebers'
]
print(len(tactical_final_list))
tactical_csv=r"C:\Users\Administrator\Desktop\Chess\player-cluster\Tactical_1_final.csv"
tactical_df=pd.read_csv(tactical_csv)
all_df=pd.read_csv(r"C:\Users\Administrator\Desktop\Chess\player-cluster\player_features.csv")

39


In [11]:
exclude_cols = ['Name', 'ELO', 'queen_promo_1LifeRatio', 'queen_promo_2LifeRatio', 
    'queen_promo_3LifeRatio', 'queen_promo_4LifeRatio', 'queen_promo_5LifeRatio', 
    'queen_promo_6LifeRatio', 'queen_promo_7LifeRatio', 'queen_promo_8LifeRatio']
feature_cols = [
    'ForceRatio',          
    'CheckRatio',         
    'CaptureRatio',       
    'PieceActivityScore', 
    
    # 马冲进中心被换掉，或者作为弃子
    'knight_1LifeRatio',    
    'knight_2LifeRatio',    
    
    # 后期战术/开放线争夺，车通常会激烈交换
    'rook_1LifeRatio',      
    'rook_2LifeRatio',
    # 象的对攻，斜线争夺
    'bishop_1LifeRatio',    
    'bishop_2LifeRatio',
    # 后的过早出动或被攻击
    'queen_1LifeRatio',     
    'queen_promo_1LifeRatio',
    # 战术发生的前提通常是有中心控制，或者在中心有接触
    'CenterControlScore'    
]
X_train = tactical_df[feature_cols].copy()
X_test = all_df[feature_cols].copy()
X_train = X_train.fillna(0)
X_test = X_test.fillna(0)

scaler = StandardScaler()
X_train_scaled=scaler.fit_transform(X_train)
X_test_scaled=scaler.transform(X_test)
#linear rbf poly sigmoid
#scale auto
ocsvm=OneClassSVM(kernel='rbf', gamma='scale', nu=0.1)
ocsvm.fit(X_train_scaled)
dist_scores = ocsvm.decision_function(X_test_scaled)
def sigmoid(x):
    return 1 / (1 + np.exp(-x))
k = 1 
prob_scores = sigmoid(k * dist_scores)
result_df = all_df.copy()

result_df['SVM_Distance'] = dist_scores  #原始距离
result_df['Similarity_Score'] = prob_scores #映射后的伪概率
#result_df['Is_Tactical_Pred'] = ocsvm.predict(X_test_scaled)

result_df = result_df.sort_values('SVM_Distance', ascending=False)
result_df['Rank_Percentile'] = result_df['SVM_Distance'].rank(pct=True)

print(result_df[['Name', 'SVM_Distance', 'Rank_Percentile']].sort_values('Rank_Percentile', ascending=False).head(10))

                        Name  SVM_Distance  Rank_Percentile
182              Ali Farahat      6.145974         1.000000
1053         Ivan Provotorov      5.815943         0.999629
2677          Николай Хныкин      5.802515         0.999258
492           ChessyInstinct      5.649090         0.998887
737           Emilio Cordova      5.641350         0.998516
1960          Raja Rithvik R      5.554598         0.998145
936         Haik Martirosyan      5.488178         0.997774
543              Daniel Chan      5.408573         0.997403
2191  Shawn Rodrigue-Lemieux      5.396106         0.997032
756               Erick Zhao      5.379639         0.996660


In [12]:
result_df['Rank_Percentile'].describe()

count    2695.000000
mean        0.500186
std         0.288729
min         0.000371
25%         0.250278
50%         0.500186
75%         0.750093
max         1.000000
Name: Rank_Percentile, dtype: float64

In [13]:
sample_df1=result_df.loc[result_df['Name'].isin(tactical_final_list),['Name', 'SVM_Distance', 'Rank_Percentile']]
print(sample_df1['Rank_Percentile'].describe())
sample_df1

count    38.000000
mean      0.695889
std       0.202019
min       0.311688
25%       0.570130
50%       0.759555
75%       0.859647
max       0.977737
Name: Rank_Percentile, dtype: float64


,Name,SVM_Distance,Rank_Percentile
1789,Novendra Priasmoro,4.463388,0.977737
2480,Volodymyr Onyshchuk,3.743058,0.949165
945,Hans Niemann,3.617294,0.941373
618,Denis Khismatullin,3.376348,0.923191
320,Arjun Erigaisi,3.371253,0.922078
553,Daniel Naroditsky,3.111872,0.898330
429,Bibisara Assaubayeva,3.066783,0.893878
562,Daniil Dubov,3.055184,0.890909
968,Hikaru Nakamura,2.839775,0.867161
1696,Mustafa Yilmaz,2.798137,0.861967


In [14]:
result_df[['Name','ELO','SVM_Distance','Similarity_Score','Rank_Percentile']].to_csv('player_style_result_tactical.csv')

In [ ]:
from sklearn.ensemble import IsolationForest

iforest = IsolationForest(
    n_estimators=300,
    max_samples='auto',
    contamination=0.05,
    random_state=42
)

iforest.fit(X_train_scaled)     # tactical only
scores = iforest.score_samples(X_test_scaled)  

In [16]:
result_df['IForest_Score'] = scores
result_df['IForest_Pct'] = pd.Series(scores).rank(pct=True)

In [17]:
result_df.head()

,Unnamed: 0,user,CenterControlScore,PieceActivityScore,KingSafetyScore,CastlingScore,KingTropismScore,KingDefendersScore,KingPawnShieldScore,KingZoneControlScore,...,knight_promo_6LifeRatio,knight_promo_7LifeRatio,knight_promo_8LifeRatio,Name,ELO,SVM_Distance,Similarity_Score,Rank_Percentile,IForest_Score,IForest_Pct
182,182,Ali Farahat,19.781637,185.287751,-55.306961,-15.156466,-11.332930,3.994611,-3.960126,-30.355144,...,NaN,NaN,NaN,Ali Farahat,2616.0,6.145974,0.997862,1.000000,-0.445717,1.000000
1053,1053,Ivan Provotorov,19.568107,186.966383,-56.397417,-13.929609,-11.295145,3.996802,-5.678881,-31.365186,...,NaN,NaN,NaN,Ivan Provotorov,2569.0,5.815943,0.997029,0.999629,-0.429424,0.998887
2677,2677,Николай Хныкин,19.490568,188.838957,-62.349646,-14.441237,-12.472463,3.977429,-5.180570,-34.933178,...,NaN,NaN,NaN,Николай Хныкин,2201.0,5.802515,0.996989,0.999258,-0.431647,0.999258
492,492,ChessyInstinct,19.249020,186.074079,-57.739905,-14.520471,-11.416524,3.993075,-5.860641,-31.367387,...,NaN,NaN,NaN,ChessyInstinct,2341.0,5.649090,0.996492,0.998887,-0.409899,0.997403
737,737,Emilio Cordova,20.162396,185.330519,-58.004906,-15.018438,-11.411224,3.997851,-5.681536,-31.357668,...,NaN,NaN,NaN,Emilio Cordova,2890.0,5.641350,0.996464,0.998516,-0.440045,0.998145


In [18]:
# IForest_Score	IForest_Pct
print(result_df[['Name', 'IForest_Score', 'IForest_Pct']].sort_values('IForest_Score', ascending=False).head(10))

                          Name  IForest_Score  IForest_Pct
699                 Ediz Gürel      -0.370004     0.788497
1180  Jose Esteban Marin Masis      -0.379852     0.315028
1033            Ismael Gimenez      -0.379967     0.354731
1562          Matthew Guo Diao      -0.380194     0.670872
951               Harry Grieve      -0.382126     0.677551
1233                Junior Tay      -0.383812     0.640445
438             Bobur Sattarov      -0.384590     0.684972
2319         Tigran Nazaretyan      -0.384873     0.757328
1533              Martin Horak      -0.386157     0.481262
1377             Levon Aronian      -0.386199     0.600000


In [19]:
result_df['IForest_Pct'].describe()

count    2695.000000
mean        0.500186
std         0.288729
min         0.000371
25%         0.250278
50%         0.500186
75%         0.750093
max         1.000000
Name: IForest_Pct, dtype: float64

In [20]:
sample_df2=result_df.loc[
    result_df['Name'].isin(tactical_final_list),
    ['Name', 'IForest_Score', 'IForest_Pct']
]
print(sample_df2['IForest_Pct'].describe())
sample_df2

count    38.000000
mean      0.733522
std       0.257650
min       0.075325
25%       0.661039
50%       0.809276
75%       0.929035
max       0.983302
Name: IForest_Pct, dtype: float64


,Name,IForest_Score,IForest_Pct
1789,Novendra Priasmoro,-0.435241,0.973284
2480,Volodymyr Onyshchuk,-0.459767,0.983302
945,Hans Niemann,-0.458936,0.963636
618,Denis Khismatullin,-0.438170,0.951391
320,Arjun Erigaisi,-0.439781,0.969944
553,Daniel Naroditsky,-0.408881,0.968460
429,Bibisara Assaubayeva,-0.424053,0.871243
562,Daniil Dubov,-0.448699,0.964750
968,Hikaru Nakamura,-0.440504,0.928757
1696,Mustafa Yilmaz,-0.444621,0.953989


In [21]:
result_df.head()

,Unnamed: 0,user,CenterControlScore,PieceActivityScore,KingSafetyScore,CastlingScore,KingTropismScore,KingDefendersScore,KingPawnShieldScore,KingZoneControlScore,...,knight_promo_6LifeRatio,knight_promo_7LifeRatio,knight_promo_8LifeRatio,Name,ELO,SVM_Distance,Similarity_Score,Rank_Percentile,IForest_Score,IForest_Pct
182,182,Ali Farahat,19.781637,185.287751,-55.306961,-15.156466,-11.332930,3.994611,-3.960126,-30.355144,...,NaN,NaN,NaN,Ali Farahat,2616.0,6.145974,0.997862,1.000000,-0.445717,1.000000
1053,1053,Ivan Provotorov,19.568107,186.966383,-56.397417,-13.929609,-11.295145,3.996802,-5.678881,-31.365186,...,NaN,NaN,NaN,Ivan Provotorov,2569.0,5.815943,0.997029,0.999629,-0.429424,0.998887
2677,2677,Николай Хныкин,19.490568,188.838957,-62.349646,-14.441237,-12.472463,3.977429,-5.180570,-34.933178,...,NaN,NaN,NaN,Николай Хныкин,2201.0,5.802515,0.996989,0.999258,-0.431647,0.999258
492,492,ChessyInstinct,19.249020,186.074079,-57.739905,-14.520471,-11.416524,3.993075,-5.860641,-31.367387,...,NaN,NaN,NaN,ChessyInstinct,2341.0,5.649090,0.996492,0.998887,-0.409899,0.997403
737,737,Emilio Cordova,20.162396,185.330519,-58.004906,-15.018438,-11.411224,3.997851,-5.681536,-31.357668,...,NaN,NaN,NaN,Emilio Cordova,2890.0,5.641350,0.996464,0.998516,-0.440045,0.998145


In [22]:
w_svm=0.6
w_if=0.4

result_df['Tactical_Style_Score'] = (result_df['Rank_Percentile'] * w_svm) + (result_df['IForest_Pct'] * w_if)
cols_to_save = ['Name', 'Tactical_Style_Score', 'Rank_Percentile', 'IForest_Pct', 'ELO']

print("最终战术风格得分 Top 10 (加权后):")
print(result_df[cols_to_save].sort_values('Tactical_Style_Score', ascending=False).head(10))

result_df[cols_to_save].to_csv('tactical_results.csv', index=False)

最终战术风格得分 Top 10 (加权后):
                        Name  Tactical_Style_Score  Rank_Percentile  \
182              Ali Farahat              1.000000         1.000000   
1053         Ivan Provotorov              0.999332         0.999629   
2677          Николай Хныкин              0.999258         0.999258   
1960          Raja Rithvik R              0.998738         0.998145   
737           Emilio Cordova              0.998367         0.998516   
492           ChessyInstinct              0.998293         0.998887   
936         Haik Martirosyan              0.998071         0.997774   
2191  Shawn Rodrigue-Lemieux              0.997032         0.997032   
756               Erick Zhao              0.996364         0.996660   
106       Alessandro Manzone              0.996289         0.996289   

      IForest_Pct     ELO  
182      1.000000  2616.0  
1053     0.998887  2569.0  
2677     0.999258  2201.0  
1960     0.999629  2919.0  
737      0.998145  2890.0  
492      0.997403  2341.0  